In [ ]:
!pip install transformers torch biopython pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 33.4 MB/s eta 0:00:00


In [ ]:
import torch
from transformers import AutoTokenizer, EsmForMaskedLM
from Bio import pairwise2
from Bio.Align import substitution_matrices
import pandas as pd

# --- SCIENTIFIC SEQUENCES ---
# DtpA (E. coli) - UniProt: P75743
dtpa_seq = "MSTANQKPTESVSLNAFKQPKAFYLIFSIELWERFGYYGLQGIMAVYLVKQLGMSEADSITLFSSFSALVYGLVAIGGWLGDKVLGTKRVIMLGAIVLAIGYTLVAWSGHDAGIVYMGMAAIAVGNGLFKANPSSLLSTCYEKNDPRLDGAFTMYYMSVNIGSFFSMIATPWLAAKYGWSVAFALSVVGLLITIVNFAFCQRWVKQYGSKPDFEPINYRNLLLTIIGVVALIAIATWLLHNQEVARMALGVVAFGIVVIFGKEAFAMKGAARRKMIVAFILMLEAIIFFVLYSQMPTSLNFFAIRNVEHSILGLAVEPEQYQALNPFWIIIGSPILAAIYNKMGDTLPMPTKFAIGMVMCSGAFLILPLGAKFASDAGIVSVSWLVASYGLQSIGELMISGLGLAMVAQLVPQRLMGFIMGSWFLTTAGANLIGGYVAGMMAVPDNVTDPLMSLEVYGRVFLQIGVATAVIAVLMLLTAPKLHRMTQDDAADKAAKAAVA" # Use full UniProt: Q5KWA5
# Human PepT1 (Target - SLC15A1)
 # Replace with full P75743 FASTA
# Human PepT1 - UniProt: P46059
hpept1_seq = "MGMSKSHSFFGYPLSIFFIVVNEFCERFSYYGMRAILILYFTNFISWDDNLSTAIYHTFVALCYLTPILGALIADSWLGKFKTIVSLSIVYTIGQAVTSVSSINDLTDHNHDGTPDSLPVHVVLSLIGLALIALGTGGIKPCVSAFGGDQFEEGQEKQRNRFFSIFYLAINAGSLLSTIITPMLRVQQCGIHSKQACYPLAFGVPAALMAVALIVFVLGSGMYKKFKPQGNIMGKVAKCIGFAIKNRFRHRSKAFPKREHWLDWAKEKYDERLISQIKMVTRVMFLYIPLPMFWALFDQQGSRWTLQATTMSGKIGALEIQPDQMQTVNAILIVIMVPIFDAVLYPLIAKCGFNFTSLKKMAVGMVLASMAFVVAAIVQVEIDKTLPVFPKGNEVQIKVLNIGNNTMNISLPGEMVTLGPMSQTNAFMTFDVNKLTRINISSPGSPVTAVTDDFKQGQRHTLLVWAPNHYQVVKDGLNQKPEKGENGIRFVNTFNELITITMSGKVYANISSYNASTYQFFPSGIKGFTISSTEIPPQCQPNFNTFYLEFGSAYTYIVQRKNDSCPEVKVFEDISANTVNMALQIPQYFLLTCGEVVFSVTGLEFSYSQAPSNMKSVLQAGWLLTVAVGNIIVLIVAGAGQFSKQWAEYILFAALLLVVCVVFAIMARFYTYINPAEIEAQFDEDEKKNRLEKSNPYFMSGANSQKQM" # Replace with full P46059 FASTA

# --- 1. SEQUENCE ALIGNMENT ---
# Using BLOSUM62 to find evolutionary equivalents
matrix = substitution_matrices.load("BLOSUM62")
alignments = pairwise2.align.globalds(dtpa_seq, hpept1_seq, matrix, -10, -0.5)
aln_dtpa, aln_hu, score, _, _ = alignments[0]

# --- 2. BINDING POCKET MAPPING ---
# Key residues in Human PepT1 known for drug binding
# (e.g., Y167, W294, E595)
human_pocket = [24, 26, 27, 31, 34, 56, 57, 60, 63, 64, 80, 75, 91, 140, 144, 162, 166, 167, 171, 282, 294, 297, 298, 328, 341, 594]
targets = []
hu_idx = 0
for i, (r_dtpa, r_hu) in enumerate(zip(aln_dtpa, aln_hu)):
    if r_hu != '-': hu_idx += 1
    if hu_idx in human_pocket and r_dtpa != '-':
        dtpa_pos = i - aln_dtpa[:i].count('-')
        targets.append({'pos': dtpa_pos, 'hu_res': r_hu, 'dtpa_res': r_dtpa})

# --- 3. ESM-2 STABILITY SCORING ---
model_name = "facebook/esm2_t33_650M_UR50D"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = EsmForMaskedLM.from_pretrained(model_name)
model.eval()

def get_fitness(seq, pos, mutant_res):
    inputs = tokenizer(seq, return_tensors="pt")
    with torch.no_grad():
        logits = model(**inputs).logits
    probs = torch.nn.functional.softmax(logits, dim=-1)
    token_id = tokenizer.encode(mutant_res, add_special_tokens=False)[0]
    return probs[0, pos + 1, token_id].item()

# --- 4. EXECUTION ---
print(f"Scientific Alignment Score: {score}")
results = []
for t in targets:
    fitness = get_fitness(dtpa_seq, t['pos'], t['hu_res'])
    results.append({
        'DtpA_Site': f"{t['dtpa_res']}{t['pos']}",
        'Human_Residue': t['hu_res'],
        'Stability_Score': fitness
    })

print(pd.DataFrame(results))

/usr/local/lib/python3.12/dist-packages/Bio/pairwise2.py:278: BiopythonDeprecationWarning: Bio.pairwise2 has been deprecated, and we intend to remove it in a future release of Biopython. As an alternative, please consider using Bio.Align.PairwiseAligner as a replacement, and contact the Biopython developers if you still need the Bio.pairwise2 module.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/724 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/95.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/93.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.61G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/571 [00:00<?, ?it/s]

EsmForMaskedLM LOAD REPORT from: facebook/esm2_t33_650M_UR50D
Key                         | Status     |  | 
----------------------------+------------+--+-
esm.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Scientific Alignment Score: 209.0
   DtpA_Site Human_Residue  Stability_Score
0        L30             F     1.611095e-04
1        E32             E     9.996504e-01
2        R33             R     9.998010e-01
3        Y37             Y     9.996258e-01
4        Q40             R     7.750429e-06
5        F62             Y     8.272395e-04
6        S63             H     3.100413e-06
7        S66             V     1.545571e-04
8        V69             C     1.252147e-03
9        Y70             Y     9.997942e-01
10       D81             D     9.997689e-01
11       T86             K     1.278857e-05
12       L97             Y     3.147948e-07
13      K129             K     9.998767e-01
14      S133             S     9.966123e-01
15      A150             F     7.227899e-07
16      Y154             F     2.993291e-02
17      Y155             Y     9.998152e-01
18      N159             N     9.996744e-01
19      A284             R     2.709767e-07
20      F301             W     3.922801e-0

In [ ]:
import torch
from transformers import AutoTokenizer, EsmForMaskedLM
import pandas as pd

# 1. Initialize the ESM-2 PLM
model_name = "facebook/esm2_t33_650M_UR50D"
print(f"Loading {model_name}...")
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = EsmForMaskedLM.from_pretrained(model_name)
model.eval()

if torch.cuda.is_available():
    model = model.to("cuda")

def get_fitness(sequence, target_pos, target_aa):
    """Calculates the PLM probability of an amino acid at a target position."""
    inputs = tokenizer(sequence, return_tensors="pt")
    if torch.cuda.is_available():
        inputs = {k: v.to("cuda") for k, v in inputs.items()}

    with torch.no_grad():
        logits = model(**inputs).logits

    probs = torch.nn.functional.softmax(logits, dim=-1)
    token_id = tokenizer.encode(target_aa, add_special_tokens=False)[0]
    return probs[0, target_pos + 1, token_id].item()

def find_epistatic_rescue(base_seq, primary_pos, primary_mut, search_range=3):
    """Sweeps neighboring residues to find a secondary mutation that improves stability."""
    seq_list = list(base_seq)
    seq_list[primary_pos] = primary_mut
    single_mutant_seq = "".join(seq_list)

    baseline_score = get_fitness(single_mutant_seq, primary_pos, primary_mut)

    amino_acids = "ACDEFGHIKLMNPQRSTVWY"
    best_rescue = None
    best_score = baseline_score

    start_search = max(0, primary_pos - search_range)
    end_search = min(len(base_seq), primary_pos + search_range + 1)

    for neighbor_pos in range(start_search, end_search):
        if neighbor_pos == primary_pos:
            continue

        original_neighbor_aa = single_mutant_seq[neighbor_pos]

        for aa in amino_acids:
            if aa == original_neighbor_aa:
                continue

            temp_list = list(single_mutant_seq)
            temp_list[neighbor_pos] = aa
            double_mutant_seq = "".join(temp_list)

            new_score = get_fitness(double_mutant_seq, primary_pos, primary_mut)

            if new_score > best_score:
                best_score = new_score
                best_rescue = {
                    'Secondary_Mutation': f"{original_neighbor_aa}{neighbor_pos+1}{aa}",
                    'New_Stability_Score': new_score,
                    'Improvement_Factor': new_score / baseline_score
                }

    return best_rescue

# --- EXECUTE THE EXPERIMENT ---
# Full DtpA Wild-Type Sequence
dtpa_seq = "MSTANQKPTESVSLNAFKQPKAFYLIFSIELWERFGYYGLQGIMAVYLVKQLGMSEADSITLFSSFSALVYGLVAIGGWLGDKVLGTKRVIMLGAIVLAIGYTLVAWSGHDAGIVYMGMAAIAVGNGLFKANPSSLLSTCYEKNDPRLDGAFTMYYMSVNIGSFFSMIATPWLAAKYGWSVAFALSVVGLLITIVNFAFCQRWVKQYGSKPDFEPINYRNLLLTIIGVVALIAIATWLLHNQEVARMALGVVAFGIVVIFGKEAFAMKGAARRKMIVAFILMLEAIIFFVLYSQMPTSLNFFAIRNVEHSILGLAVEPEQYQALNPFWIIIGSPILAAIYNKMGDTLPMPTKFAIGMVMCSGAFLILPLGAKFASDAGIVSVSWLVASYGLQSIGELMISGLGLAMVAQLVPQRLMGFIMGSWFLTTAGANLIGGYVAGMMAVPDNVTDPLMSLEVYGRVFLQIGVATAVIAVLMLLTAPKLHRMTQDDAADKAAKAAVA"

# Your risky targets (0-indexed for Python: Position - 1)
risky_targets = [
    {'name': 'S64H', 'pos': 63, 'mut': 'H'},
    {'name': 'L98Y', 'pos': 97, 'mut': 'Y'},
    {'name': 'A285R', 'pos': 284, 'mut': 'R'},
    {'name': 'F302W', 'pos': 301, 'mut': 'W'},
    {'name': 'R305F', 'pos': 304, 'mut': 'F'},
    {'name': 'A337D', 'pos': 336, 'mut': 'D'}
]

print("\nRunning Epistatic Rescue Sweep...")
final_results = []

for target in risky_targets:
    print(f"Analyzing {target['name']}...")
    rescue_data = find_epistatic_rescue(dtpa_seq, target['pos'], target['mut'], search_range=3)

    if rescue_data:
        final_results.append({
            'Target_Site': target['name'],
            'Best_Rescue_Mutation': rescue_data['Secondary_Mutation'],
            'New_Score': f"{rescue_data['New_Stability_Score']:.2e}",
            'Fold_Improvement': f"{rescue_data['Improvement_Factor']:.1f}x"
        })
    else:
        final_results.append({
            'Target_Site': target['name'],
            'Best_Rescue_Mutation': "None found in range",
            'New_Score': "N/A",
            'Fold_Improvement': "N/A"
        })

df_final = pd.DataFrame(final_results)
print("\n--- EPISTATIC RESCUE RESULTS ---")
print(df_final.to_string(index=False))

Loading facebook/esm2_t33_650M_UR50D...


Loading weights:   0%|          | 0/571 [00:00<?, ?it/s]

EsmForMaskedLM LOAD REPORT from: facebook/esm2_t33_650M_UR50D
Key                         | Status     |  | 
----------------------------+------------+--+-
esm.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Running Epistatic Rescue Sweep...
Analyzing S64H...
Analyzing L98Y...
Analyzing A285R...
Analyzing F302W...
Analyzing R305F...
Analyzing A337D...

--- EPISTATIC RESCUE RESULTS ---
Target_Site Best_Rescue_Mutation New_Score Fold_Improvement
       S64H                 T61Y  7.02e-03            46.4x
       L98Y                 A95L  3.63e-05             2.9x
      A285R                E284I  3.08e-06             6.0x
      F302W                A303G  6.07e-03             2.3x
      R305F                I304K  4.02e-02            22.1x
      A337D                A338R  1.29e-05             4.9x


In [ ]:
import re

def validate_epistasis(base_seq, secondary_mut_str):
    """
    Validates if a secondary mutation is a true epistatic rescuer by testing
    its fitness independently in the wild-type background.

    Args:
        base_seq: The wild-type sequence.
        secondary_mut_str: The mutation string (e.g., 'Y38D').

    Returns:
        is_epistatic (bool): True if the mutation is NOT independently stabilizing.
        independent_fitness (float): ESM-2 score of the mutation in WT background.
        wt_fitness (float): ESM-2 score of the original amino acid.
    """
    # Parse the mutation string (e.g., "Y38D" -> orig="Y", pos=38, new="D")
    match = re.match(r"([A-Z])(\d+)([A-Z])", secondary_mut_str)
    if not match:
        return False, 0.0, 0.0

    orig_aa = match.group(1)
    pos = int(match.group(2)) - 1  # Convert to 0-indexed for Python
    new_aa = match.group(3)

    # 1. Calculate fitness of the original amino acid in the WT sequence
    wt_fitness = get_fitness(base_seq, pos, orig_aa)

    # 2. Calculate fitness of the NEW amino acid in the WT sequence (No primary mutation)
    independent_fitness = get_fitness(base_seq, pos, new_aa)

    # 3. Epistasis check: It is true epistasis if it does NOT significantly
    # outperform the wild-type residue on its own.
    is_epistatic = independent_fitness <= wt_fitness

    return is_epistatic, independent_fitness, wt_fitness

In [ ]:
print("\nRunning Epistatic Rescue Sweep & Validation...")
final_results = []

for target in risky_targets:
    print(f"Analyzing {target['name']}...")

    # Step 1: Find the best compensatory mutation
    rescue_data = find_epistatic_rescue(dtpa_seq, target['pos'], target['mut'], search_range=3)

    if rescue_data:
        sec_mut = rescue_data['Secondary_Mutation']

        # Step 2: Validate compensatory epistasis (Secondary Test)
        is_epistatic, ind_fit, wt_fit = validate_epistasis(dtpa_seq, sec_mut)

        final_results.append({
            'Target_Site': target['name'],
            'Best_Rescue_Mutation': sec_mut,
            'New_Score': f"{rescue_data['New_Stability_Score']:.2e}",
            'Fold_Improvement': f"{rescue_data['Improvement_Factor']:.1f}x",
            'Is_True_Epistasis': is_epistatic,
            'WT_Background_Score': f"{wt_fit:.2e}",
            'Independent_Mut_Score': f"{ind_fit:.2e}"
        })
    else:
        final_results.append({
            'Target_Site': target['name'],
            'Best_Rescue_Mutation': "None found in range",
            'New_Score': "N/A",
            'Fold_Improvement': "N/A",
            'Is_True_Epistasis': "N/A",
            'WT_Background_Score': "N/A",
            'Independent_Mut_Score': "N/A"
        })

df_final = pd.DataFrame(final_results)
print("\n--- EPISTATIC RESCUE & VALIDATION RESULTS ---")
print(df_final.to_string(index=False))


Running Epistatic Rescue Sweep & Validation...
Analyzing S64H...
Analyzing L98Y...
Analyzing A285R...
Analyzing F302W...
Analyzing R305F...
Analyzing A337D...

--- EPISTATIC RESCUE & VALIDATION RESULTS ---
Target_Site Best_Rescue_Mutation New_Score Fold_Improvement  Is_True_Epistasis WT_Background_Score Independent_Mut_Score
       S64H                 T61Y  7.02e-03            46.4x               True            9.20e-01              1.18e-05
       L98Y                 A95L  3.63e-05             2.9x               True            9.98e-01              8.50e-06
      A285R                E284I  3.08e-06             6.0x               True            9.72e-01              3.35e-05
      F302W                A303G  6.07e-03             2.3x               True            9.92e-01              1.35e-04
      R305F                I304K  4.02e-02            22.1x               True            9.90e-01              4.49e-05
      A337D                A338R  1.29e-05             4.9x        